### **Faiss**

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter


# Load document
loader = TextLoader("my_file.txt")
documents = loader.load()


# Split into chunks
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)


# Create embeddings
embeddings = OpenAIEmbeddings()


# Create FAISS index
vectorstore = FAISS.from_documents(docs, embeddings)


# Save locally
vectorstore.save_local("faiss_index")

In [ ]:
# Load an Existing Index

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

#### **Flat Index**

In [ ]:
# Index
dimension = 1534

index = FAISS.IndexFlatL2(dimension)

vectorstore = FAISS(
    embedding_function=embeddings,
    index=index,
    normalize_L2=True
    )

#### **🚀 Approximate Nearest Neighbor (ANN)**

ANN trades tiny accuracy loss for massive speed gains.


#### 2️⃣ **IVF (Inverted File Index)**

 **`IndexIVFFlat`**

**How it works**

- Clusters vectors into `nlist` groups  
- Searches only closest clusters  

**Important Parameters**

- `nlist` → number of clusters  
- `nprobe` → how many clusters to search at query time  

**Tradeoff**

- Higher `nprobe` = more accurate but slower  
- Lower `nprobe` = faster but less accurate  

**Best for**

- 1M+ vectors  
- Large-scale production search  

**Example**

In [ ]:
d = 1534
training_vectors = []
vectors = []

quantizer = FAISS.IndexFlatL2(d)
index = FAISS.IndexIVFFlat(quantizer, d, nlist=100)
index.train(training_vectors)
index.add(vectors)

index.nprobe = 10

#### 3️⃣ **HNSW (Graph-based ANN)**

**`IndexHNSWFlat`**

Very popular modern ANN method.

**How it works**

- Builds a navigable small-world graph  
- Fast query times  
- High recall  

**Key Parameters**

- `M` → number of connections per node (higher = more memory, better accuracy)  
- `efSearch` → search accuracy/speed tradeoff  

**Best for**

- Real-time systems  
- Balanced speed + accuracy  
- Often better than IVF for many workloads  

**Example**

In [ ]:
import faiss

index = faiss.IndexHNSWFlat(d, M=32)
index.hnsw.efSearch = 50

#### 4️⃣ **PQ (Product Quantization)**

**`IndexIVFPQ`**

Compresses vectors to reduce memory usage.

- Much smaller memory footprint  
- Slightly lower accuracy  
- Great for very large datasets (10M+)  

**Best when**

- Memory constrained  
- Massive vector stores  

---

##### 📊 **Quick Comparison**

| Index Type | Accuracy | Speed | Memory | Scale   |
|------------|----------|-------|--------|---------|
| Flat       | ⭐⭐⭐⭐⭐   | ⭐     | High   | Small   |
| IVF        | ⭐⭐⭐⭐    | ⭐⭐⭐⭐  | Medium | Large   |
| HNSW       | ⭐⭐⭐⭐½   | ⭐⭐⭐⭐⭐ | Medium-High | Large |
| IVFPQ      | ⭐⭐⭐     | ⭐⭐⭐⭐  | Low    | Massive |

---

#### 🧠 **Which Should You Use?**

##### 🔹 < 500k vectors

Use **Flat** or **HNSW**

##### 🔹 1M–10M vectors

Use **HNSW** or **IVF**

##### 🔹 10M+ vectors

Use **IVFPQ**

---

##### ⚙️ **Using Custom FAISS Index in LangChain**

LangChain lets you pass a pre-built FAISS index:

In [ ]:
from langchain_community.vectorstores import FAISS

custom_index = FAISS.indexHNSW()
vectorstore = FAISS(
    embedding_function=embeddings,
    index=custom_index
)

You must:

- Create and train the FAISS index yourself  
- Then wrap it in LangChain  

---

#### 🔥 **Performance Tuning Tips**

##### IVF

- `nlist ≈ sqrt(N)`  
- `nprobe` between 5–20 to start  

##### HNSW

- `M = 16–48`  
- `efSearch = 40–200`  
- Higher `efConstruction` improves quality at build time  

---

#### 🧪 **Production Advice**

If you're building:

- RAG chatbot → HNSW is often best  
- Offline batch search → IVF  
- Memory-limited environment → IVFPQ  
- Research / perfect recall → Flat  